# SymFormer / TBX11K — Colab Runbook (Core-Method PoC)

Executes the phases in `plan.md` on a free Colab **T4**. Run cells top-to-bottom.

**Stack: torch + torchvision only.** No mmcv/mmdetection — OpenMMLab ships wheels only up to
~torch 2.1 / Python 3.11, and Colab is on Python 3.12, so `mim install mmcv` falls into a source
build that fails. torchvision gives the same ResNet-50 + FPN + RetinaNet architecture, is
preinstalled, and needs **zero installs**. Our SAS block is pure torch and is unchanged.

**Storage split** (free Drive is 15GB; the Colab VM has ~100GB of ephemeral disk):

| What | Where | Size |
|---|---|---|
| code / configs / docs | **GitHub** → `/content/FOP` | ~200 KB |
| raw TBX11K | **`/content`** (ephemeral — never Drive) | ~11–30 GB |
| compact TB-only 512² dataset | **Drive** | ~250–400 MB |
| checkpoints + logs | **Drive** | ~600 MB (+~300 MB transient per ablation cell) |

Training cells auto-resume, so after a time-out just re-run the same cell.
See `paper.md` (method + target numbers), `CLAUDE.md` (scope/recipe), `limitations.md` (caveats).

## Phase 1 — environment

In [ ]:
!nvidia-smi   # confirm a GPU (ideally a T4) is attached; if none, change runtime or retry later

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
# ---- paths ----
# Storage split: free Drive is only 15GB, but the Colab VM has ~100GB of EPHEMERAL local disk.
#   raw TBX11K (tens of GB) -> /content  (download, prep, then let it die with the session)
#   compact 512 dataset + checkpoints/logs -> Drive (must survive time-outs)
# Expected Drive usage: ~1.5GB peak.
REPO_URL = 'https://github.com/Manas-Maahir/FOP.git'
PROJECT  = '/content/FOP'                 # this repo on Colab
DRIVE    = '/content/drive/MyDrive'
DATA_RAW = '/content/TBX11K'              # raw TBX11K — EPHEMERAL, deliberately NOT on Drive
DATA_512 = f'{DRIVE}/tbx11k_tb512'        # compact TB-only 512 dataset (built in Phase 3)
RUNS     = f'{DRIVE}/tb_runs'             # checkpoints + logs (persist across sessions)
os.makedirs(RUNS, exist_ok=True)
print('project       :', PROJECT)
print('raw (temp)    :', DATA_RAW)
print('compact(Drive):', DATA_512)
print('runs (Drive)  :', RUNS)

In [ ]:
# Pull the code from GitHub. Re-running this in a later session picks up anything pushed since.
if os.path.isdir(PROJECT):
    !cd {PROJECT} && git pull --ff-only
else:
    !git clone {REPO_URL} {PROJECT}
%cd {PROJECT}
!ls

In [ ]:
# No mmcv/mmdet. torch + torchvision are already on Colab; pycocotools usually is too.
# DO NOT `pip install -U openmim` — it downgrades setuptools and breaks on Python 3.12.
import torch, torchvision, sys
print('python     ', sys.version.split()[0])
print('torch      ', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')
print('torchvision', torchvision.__version__)
try:
    import pycocotools; print('pycocotools OK')
except ImportError:
    !pip install -q pycocotools
    import pycocotools; print('pycocotools installed')
# freeze versions (RECORD these in results.md)
open('requirements.lock.txt','w').write(
    f"python=={sys.version.split()[0]}\ntorch=={torch.__version__}\n"
    f"torchvision=={torchvision.__version__}\n")
print('wrote requirements.lock.txt')

## Phase 6 (early) — unit tests
These gate the model code. Run them **before** downloading tens of GB of data — they need no data.

In [ ]:
!python tests/test_numpy_ref.py    # mirror + PE math (torch-free)
!python tests/test_sas.py          # the 4 required SAS tests + numpy cross-checks
!python tests/test_tv_model.py     # RetinaNet + SAS wiring, weight sharing, train/infer

## Phases 2–3 — data
Download the raw TBX11K to **ephemeral `/content`** (NOT Drive), build the compact TB-only 512²
set, and write only that to Drive. Later sessions need only the compact set.

In [ ]:
# sanity-check the prep logic first (synthetic, no real data):
!python tools/prepare_tbx11k.py --selftest

In [ ]:
# Skip the download entirely if the compact set is already on Drive.
import os
if os.path.isdir(DATA_512):
    print('compact dataset already on Drive ->', DATA_512, '(skip download/prep)')
else:
    os.makedirs(DATA_RAW, exist_ok=True)
    print('Download TBX11K into', DATA_RAW, 'using the links at')
    print('  https://github.com/yun-liu/Tuberculosis')
    print('e.g.  !gdown <FILE_ID> -O /content/tbx11k.zip && !unzip -q /content/tbx11k.zip -d /content')
    print('Raw data stays on /content and dies with the session — that is intentional.')
!df -h /content | tail -1   # free ephemeral disk

In [ ]:
# Inspect the layout BEFORE prepping — the flags below assume annotations/xml, imgs/, lists/.
# If your copy differs, adjust the flags (and DEFAULT_CATMAP in tools/prepare_tbx11k.py).
!ls {DATA_RAW} 2>/dev/null && echo '--- annotations ---' && ls {DATA_RAW}/annotations 2>/dev/null \
  && echo '--- lists ---' && ls {DATA_RAW}/lists 2>/dev/null

In [ ]:
# Build the compact TB-only 512 COCO dataset -> Drive.
# GATE: expect TB train ~600 and val ~200 images (paper Table 2).
!python tools/prepare_tbx11k.py --src {DATA_RAW} --dst {DATA_512} \
    --xml-dir annotations/xml --img-dir imgs \
    --train-list {DATA_RAW}/lists/TB_train.txt --val-list {DATA_RAW}/lists/TB_val.txt \
    --size 512 --write-agnostic
!du -sh {DATA_512}   # expect a few hundred MB

## Phase 4 — smoke test (a few batches, 1 epoch)

In [ ]:
!python tools/tv_train.py --work-dir {RUNS}/smoke --data-root {DATA_512}/ --no-sas \
    --epochs 1 --batch-size 2 --limit-batches 5
!python tools/tv_eval.py --ckpt {RUNS}/smoke/epoch_1.pth --data-root {DATA_512}/
# GATE: runs end-to-end and prints a finite AP (it will be ~0 — that is fine).

## Phase 5 — RetinaNet baseline (re-run after any time-out; it auto-resumes)

In [ ]:
!python tools/tv_train.py --work-dir {RUNS}/baseline --data-root {DATA_512}/ --no-sas \
    --epochs 24 --batch-size 8 --lr 0.005 --seed 0

In [ ]:
!python tools/tv_eval.py --ckpt {RUNS}/baseline/epoch_24.pth --data-root {DATA_512}/

## Phase 7 — SymFormer w/ RetinaNet (the primary comparison)

In [ ]:
!python tools/tv_train.py --work-dir {RUNS}/symformer --data-root {DATA_512}/ \
    --attention symattention --pe spe --stn --direction r2l \
    --epochs 24 --batch-size 8 --lr 0.005 --seed 0

In [ ]:
!python tools/tv_eval.py --ckpt {RUNS}/symformer/epoch_24.pth --data-root {DATA_512}/
# GATE (primary claim): SymFormer AP50 > baseline AP50. Record both in results.md.

## Phase 8 — Table 8 ablation (long; each cell auto-resumes)
Checkpoints are deleted after each cell is evaluated so Drive stays bounded.

In [ ]:
import subprocess, glob, os, json
DELETE_CKPTS_AFTER_EVAL = True   # False keeps every ablation checkpoint (~5GB)
EPOCHS = 24

# (name, extra flags) — most-informative cells first, so a partial run is still meaningful.
CELLS = [
    ('none_none',            ['--no-sas']),                                                  # baseline
    ('vanilla_ape',          ['--attention','vanilla','--pe','ape']),
    ('symattention_ape',     ['--attention','symattention','--pe','ape']),
    ('symattention_spe_stn_r2l',   ['--attention','symattention','--pe','spe','--stn','--direction','r2l']),
    ('symattention_spe_nostn_r2l', ['--attention','symattention','--pe','spe','--no-stn','--direction','r2l']),
    ('symattention_spe_stn_l2r',   ['--attention','symattention','--pe','spe','--stn','--direction','l2r']),
    ('vanilla_rpe',          ['--attention','vanilla','--pe','rpe']),
    ('symattention_rpe',     ['--attention','symattention','--pe','rpe']),
    ('vanilla_spe_nostn_l2r',['--attention','vanilla','--pe','spe','--no-stn','--direction','l2r']),
    ('vanilla_spe_nostn_r2l',['--attention','vanilla','--pe','spe','--no-stn','--direction','r2l']),
    ('vanilla_spe_stn_l2r',  ['--attention','vanilla','--pe','spe','--stn','--direction','l2r']),
    ('vanilla_spe_stn_r2l',  ['--attention','vanilla','--pe','spe','--stn','--direction','r2l']),
    ('symattention_spe_nostn_l2r', ['--attention','symattention','--pe','spe','--no-stn','--direction','l2r']),
]

for name, flags in CELLS:
    wd = f'{RUNS}/abl/{name}'
    print('='*70); print('TRAIN', name); print('='*70)
    subprocess.run(['python','tools/tv_train.py','--work-dir',wd,'--data-root',f'{DATA_512}/',
                    '--epochs',str(EPOCHS),'--batch-size','8','--lr','0.005','--seed','0'] + flags)
    ck = f'{wd}/epoch_{EPOCHS}.pth'
    if os.path.isfile(ck):
        print('EVAL', name)
        subprocess.run(['python','tools/tv_eval.py','--ckpt',ck,'--data-root',f'{DATA_512}/',
                        '--tag',name])
        if DELETE_CKPTS_AFTER_EVAL:
            for p in glob.glob(f'{wd}/*.pth'):
                os.remove(p)
            print('removed checkpoints for', name)
    else:
        print('!! no checkpoint for', name, '- training did not finish; re-run this cell to resume')
!du -sh {RUNS}

In [ ]:
# Collect every eval_log.jsonl into one table for results.md
import glob, json
rows = []
for f in sorted(glob.glob(f'{RUNS}/**/eval_log.jsonl', recursive=True)):
    for line in open(f):
        rows.append(json.loads(line))
print(f"{'run':32s} {'AP50':>6s} {'AP':>6s}")
for r in rows:
    print(f"{r['tag']:32s} {r['AP50']:6.1f} {r['AP']:6.1f}")

## Phase 10 — write-up
Fill `results.md` (baseline vs SymFormer + the ablation table), update `CLAUDE.md` §0/§6/§8 with
the actual numbers, and note deviations (torchvision instead of mmdet, batch size, versions) and
caveats (val-not-test; reduced data → trend not absolute numbers). Then commit and push.